In [8]:
import torch
import torch.nn as nn
import numpy as np
# np.random.seed(42)
x=torch.tensor([1,2,3,4,5,6,7,8,9,10], dtype=torch.float32)
y=torch.tensor([0,1,0,1,0,1,0,1,0,1], dtype=torch.float32)
print(x)
print(y)


tensor([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.])
tensor([0., 1., 0., 1., 0., 1., 0., 1., 0., 1.])


In [9]:
class SimpleNeuralNetwork:
    def __init__(self):
        self.weights = torch.randn(1,requires_grad=True)
        self.bias = torch.randn(1,requires_grad = True)
    def forward(self,x):
        return self.weights *x +self.bias
    def loss(self,y_pred,y):
        loss = nn.MSELoss()
        return loss(y_pred,y)

In [10]:
model = SimpleNeuralNetwork()
cri=nn.MSELoss()

print(model)
print(model.weights)
print(model.bias)
# print(loss)

tensor([2.3328], requires_grad=True)
tensor([0.3108], requires_grad=True)


In [11]:
#training loop
for _ in range(10):
    y_pred =model.forward(x)
    # loss=model.loss(y_pred,y)
    # print(loss)
    loss =cri(y_pred,y)
    
    loss.backward()
    with torch.no_grad():
        model.weights -=0.01* model.weights.grad
        model.bias -= 0.01*model.bias.grad
    model.weights.grad.zero_()
    model.bias.grad.zero_()
    print(loss)

tensor(203.7782, grad_fn=<MseLossBackward0>)
tensor(9.6052, grad_fn=<MseLossBackward0>)
tensor(0.6961, grad_fn=<MseLossBackward0>)
tensor(0.2871, grad_fn=<MseLossBackward0>)
tensor(0.2682, grad_fn=<MseLossBackward0>)
tensor(0.2671, grad_fn=<MseLossBackward0>)
tensor(0.2668, grad_fn=<MseLossBackward0>)
tensor(0.2666, grad_fn=<MseLossBackward0>)
tensor(0.2664, grad_fn=<MseLossBackward0>)
tensor(0.2662, grad_fn=<MseLossBackward0>)


In [12]:
import numpy as np
import cv2
import os


os.makedirs("dataset/square",exist_ok=True)
os.makedirs("dataset/circle",exist_ok=True)

img_size=128

def generate_square():
    img =np.zeros((img_size,img_size),dtype=np.uint8)
    start = np.random.randint(10,30)
    size = np.random.randint(20,50)
    img[start:start+size,start:start+size]=255
    return img
def generate_circle():
    img = np.zeros((img_size,img_size),dtype = np.uint8)
    center = (np.random.randint(30,98),np.random.randint(30,98))
    radius = np.random.randint(10,30)
    cv2.circle(img,center,radius,255,-1)

    return img
for i in range(30):
    square_img = generate_square()
    cv2.imwrite(f"dataset/square/square_{i}.png",square_img)
    circle_img = generate_circle()
    cv2.imwrite(f"dataset/circle/circle_{i}.png",circle_img)


In [13]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN,self).__init__()
        self.conv1 =nn.Conv2d(1,16,3,padding=1)
        self.conv2 =nn.Conv2d(16,32,3,padding=1)
        self.fc1 = nn.Linear(32*128*128,64)
        self.fc2 = nn.Linear(64,2)
    def forward(self,x):
        x=torch.relu(self.conv1(x))
        # print(x.shape)
        x=torch.relu(self.conv2(x))
        # print(x.shape)
        x=x.view(x.size(0),-1)
        # print(x.shape)
        x=torch.relu(self.fc1(x))
        # print(x.shape)
        x=self.fc2(x)
        # print(x.shape)
        return x
# import torch.nn as nn
# import torch.nn.functional as F

# class SimpleCNN(nn.Module):
#     def __init__(self):
#         super().__init__()

#         self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
#         self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
#         self.pool = nn.MaxPool2d(2, 2)

#         self.fc1 = nn.Linear(32 * 256, 64)
#         self.fc2 = nn.Linear(64, 2)

#         self.flatten = nn.Flatten()

#     def forward(self, x):
#         print(x.shape)
#         x = self.pool(F.relu(self.conv1(x)))  # 64→32
#         print(x.shape)
#         x = self.pool(F.relu(self.conv2(x)))  # 32→16
#         print(x.shape)
#         x = self.flatten(x)  # 16*16*32
#         print(x.shape)
#         x = F.relu(self.fc1(x))
#         print(x.shape)
#         x = self.fc2(x)
#         print(x.shape)
#         return x



In [18]:
model=SimpleCNN()
print(model)
print(model.forward(torch.randn(8,1,128,128)))


SimpleCNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=524288, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=2, bias=True)
)
tensor([[0.0395, 0.0402],
        [0.0254, 0.0790],
        [0.0332, 0.0789],
        [0.0787, 0.0486],
        [0.0625, 0.0890],
        [0.0252, 0.0785],
        [0.0480, 0.0718],
        [0.0046, 0.0805]], grad_fn=<AddmmBackward0>)


In [22]:
import torch
from torch.utils.data import Dataset, DataLoader,random_split
from torchvision import transforms
from PIL import Image
import glob

class ShapeDataset(Dataset):
    def __init__(self, root):
        self.files = glob.glob(root + "/*/*.png")
        self.transform = transforms.Compose([
            transforms.Grayscale(),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_path = self.files[idx]
        image = Image.open(img_path)
        image = self.transform(image)

        label = 0 if "square" in img_path else 1
        return image, torch.tensor(label, dtype=torch.long)

dataset = ShapeDataset("dataset")
loader = DataLoader(dataset, batch_size=8, shuffle=True)
num_samples = dataset.__len__()
train_set,val_set,test_set = random_split(dataset,[int(0.8*num_samples), int(0.1*num_samples), int(0.1*num_samples)])
#8,1,128,128
train_loader = DataLoader(train_set, batch_size=8, shuffle=True)
val_loader = DataLoader(val_set, batch_size=8, shuffle=True)
test_loader = DataLoader(test_set, batch_size=8, shuffle=True)
print(dataset.__len__())

60


In [ ]:
model = SimpleCNN()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

Epochs =10
for epoch in range(Epochs):
    losses = 0
    for images,labels in train_loader:
        # print(images.shape)
        out = model(images)
        loss = criterion(out,labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses += loss.item()
    print(f"Epoch {epoch+1}, Loss: {losses/len(loader)}")

    with torch.no_grad():
        correct =0
        total=0
        for images,labels in val_loader:
            out = model(images)
            _,predicted = torch.max(out.data,1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        print(f"Validation Accuracy: {100*correct/total:.2f}%")


torch.Size([8, 1, 128, 128])
torch.Size([8, 1, 128, 128])
torch.Size([8, 1, 128, 128])
torch.Size([8, 1, 128, 128])
torch.Size([8, 1, 128, 128])
torch.Size([8, 1, 128, 128])
Epoch 1, Loss: 6.595418848097324
Validation Accuracy: 16.67%
torch.Size([8, 1, 128, 128])
torch.Size([8, 1, 128, 128])
torch.Size([8, 1, 128, 128])
torch.Size([8, 1, 128, 128])
torch.Size([8, 1, 128, 128])
torch.Size([8, 1, 128, 128])
Epoch 2, Loss: 1.0951048359274864
Validation Accuracy: 83.33%
torch.Size([8, 1, 128, 128])
torch.Size([8, 1, 128, 128])
torch.Size([8, 1, 128, 128])
torch.Size([8, 1, 128, 128])
torch.Size([8, 1, 128, 128])
torch.Size([8, 1, 128, 128])
Epoch 3, Loss: 0.32885378412902355
Validation Accuracy: 83.33%
torch.Size([8, 1, 128, 128])
torch.Size([8, 1, 128, 128])
torch.Size([8, 1, 128, 128])
torch.Size([8, 1, 128, 128])
torch.Size([8, 1, 128, 128])
torch.Size([8, 1, 128, 128])
Epoch 4, Loss: 0.19042605720460415
Validation Accuracy: 100.00%
torch.Size([8, 1, 128, 128])
torch.Size([8, 1, 128, 12

In [27]:
model.eval()
with torch.no_grad():
    correct =0
    total=0
    for images,labels in test_loader:
        out = model(images)
        _,predicted = torch.max(out.data,1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    print(f"Test Accuracy: {100*correct/total:.2f}%")

Test Accuracy: 100.00%
